# Квантизация fp16 → int4 и матмул bf16 × int4 на Triton

Ноутбук реализует два ядра Triton и набор проверок. Все ячейки запускаются сверху вниз; ничего не пишется в файлы.

**Задача 1 — квантизация.** `W (M, N) fp16 → int4`, симметричная: `scale = max|W| / 7`, `q = clamp(round(W / scale), -8, 7)`, ноль остаётся нулём. Два варианта scale: per-tensor (`scale` формы `(1,)`) и per-row (`(M,)`). Упаковка в `int8` (2 значения) или `int32` (8 значений), параметр `pack_bits ∈ {8, 32}`. Чётный индекс — младшие 4 бита, нечётный — старшие, знак в двоичном дополнении. Память ровно в 4× меньше fp16.

**Задача 2 — матмул.** `out = X @ W4ᵀ`, `X (M, K) bf16`, `W4` — упакованная `(N, K)` int4, `out (M, N) bf16`. Распаковка nibble'ов битовыми сдвигами прямо в ядре, накопление в fp32, умножение на `scale` в эпилоге.

**Ограничения и допущения**
- Каждая строка упаковывается независимо и дополняется нулями до кратности 2 (int8) или 8 (int32). «Ровно 4×» выполняется строго при `N % 2 == 0` (int8) и `N % 8 == 0` (int32); иначе размер равен `M · ceil(N / ELEMS)` слов.
- Округление — half-up: `floor(x + 0.5)`. Ядро и torch-эталон используют одни и те же fp32-операции (умножение на `1/scale`, а не деление), поэтому результаты совпадают бит-в-бит.
- На GPU старше Ampere (T4, sm_75) нет bf16 tensor cores: ядро автоматически считает `tl.dot` в fp32. Результат корректен, скорость ниже.

In [1]:
# Установка зависимостей, проверка CUDA, seed
import importlib.util
import random
import subprocess
import sys

if importlib.util.find_spec("triton") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton"])

import torch
import triton

SEED = 0
BF16_MIN_CAPABILITY_MAJOR = 8              # bf16 tensor cores: Ampere (sm_80) и новее

random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA недоступна: Runtime → Change runtime type → GPU (T4 / L4 / A100)")
torch.cuda.manual_seed_all(SEED)

_major, _minor = torch.cuda.get_device_capability()
print(f"torch {torch.__version__} | triton {triton.__version__} | CUDA {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{_major}{_minor})")
if _major < BF16_MIN_CAPABILITY_MAJOR:
    print("Внимание: bf16 tensor cores нет — матмул пойдёт через fp32-путь (корректно, но медленнее).")

torch 2.11.0+cu130 | triton 3.6.0 | CUDA 13.0
GPU: Tesla T4 (sm_75)
Внимание: bf16 tensor cores нет — матмул пойдёт через fp32-путь (корректно, но медленнее).


In [2]:
# Общие константы и эталонные torch-утилиты (упаковка, распаковка, деквантизация)
"""Общие константы и эталонные (чистый torch) утилиты для int4-квантизации.

Содержит:
  * именованные константы формата int4 и упаковки (никаких «магических» чисел в ядрах);
  * вычисление scale (per-tensor / per-row) для симметричной квантизации;
  * эталонные квантизатор, упаковщик и распаковщик на torch (для валидации ядер);
  * dequantize_int4_to_fp16 — обратное преобразование (torch, без отдельного ядра);
  * проверку «ровно 4x» по памяти.

Формат упакованных данных (layout):
  Матрица (R, C) хранится построчно (row-major). Каждая строка независимо
  дополняется нулевыми nibble'ами до кратности ELEMS = PACK_BITS // 4, поэтому
  упакованный тензор имеет форму (R, ceil(C / ELEMS)) и dtype int8 (PACK_BITS=8)
  или int32 (PACK_BITS=32). Внутри слова элемент с индексом e (0..ELEMS-1)
  лежит в битах [4*e, 4*e + 4): чётный индекс -> младшие 4 бита байта,
  нечётный -> старшие 4 бита. Значение хранится в 4-битном двоичном дополнении.
"""

import torch

# --------------------------------------------------------------------------
# Константы формата int4
# --------------------------------------------------------------------------
INT4_BITS = 4                      # ширина одного значения в битах
INT4_MIN = -8                      # минимум знакового int4
INT4_MAX = 7                       # максимум знакового int4
SYMMETRIC_QMAX = 7                 # scale = max|W| / 7 (симметричный диапазон [-7, 7])
NIBBLE_MASK = 0xF                  # маска младших 4 бит
NIBBLE_SIGN_BIT = 0x8              # знаковый бит nibble'а
ROUND_HALF = 0.5                   # округление «half up»: floor(x + 0.5)
UNIT_SCALE = 1.0                   # scale для нулевого тензора (избегаем деления на 0)

# --------------------------------------------------------------------------
# Константы упаковки
# --------------------------------------------------------------------------
PACK_BITS_INT8 = 8                 # 2 значения int4 в одном int8
PACK_BITS_INT32 = 32               # 8 значений int4 в одном int32
SUPPORTED_PACK_BITS = (PACK_BITS_INT8, PACK_BITS_INT32)
BITS_PER_BYTE = 8
FP16_BITS = 16
EXPECTED_COMPRESSION = FP16_BITS // INT4_BITS   # fp16 -> int4: ровно 4x

DEFAULT_SEED = 0


def cdiv(a: int, b: int) -> int:
    """Целочисленное деление с округлением вверх.

    Вход: a, b — целые, b > 0. Выход: ceil(a / b).
    """
    return (a + b - 1) // b


def validate_pack_bits(pack_bits: int) -> None:
    """Проверяет, что pack_bits ∈ {8, 32}; иначе бросает ValueError."""
    if pack_bits not in SUPPORTED_PACK_BITS:
        raise ValueError(f"pack_bits должен быть одним из {SUPPORTED_PACK_BITS}, получено {pack_bits}")


def elems_per_word(pack_bits: int) -> int:
    """Сколько значений int4 помещается в одно упакованное слово.

    Вход: pack_bits ∈ {8, 32}. Выход: 2 (для int8) или 8 (для int32).
    """
    validate_pack_bits(pack_bits)
    return pack_bits // INT4_BITS


def packed_torch_dtype(pack_bits: int) -> torch.dtype:
    """torch-dtype упакованного тензора: int8 для pack_bits=8, int32 для pack_bits=32."""
    validate_pack_bits(pack_bits)
    return torch.int8 if pack_bits == PACK_BITS_INT8 else torch.int32


def compute_scale(W: torch.Tensor, per_row: bool = False):
    """Считает scale симметричной квантизации и обратный scale (оба float32).

    Вход:
      W       — (R, C) fp16 (или любой float) тензор на любом устройстве;
      per_row — False: один scale на тензор, форма (1,); True: scale на строку, форма (R,).
    Выход: (scale, inv_scale) — float32, scale = max|W| / 7, inv_scale = 1 / scale.
    Ограничения: если max|W| == 0, scale = 1.0 (все коды будут нулями).
    Замечание: ядро умножает на inv_scale (а не делит) — так результат бит-в-бит
    совпадает с torch-эталоном (IEEE-умножение, без приближённого деления).
    """
    abs_w = W.abs()
    if per_row:
        amax = abs_w.amax(dim=1).to(torch.float32)
    else:
        amax = abs_w.amax().to(torch.float32).reshape(1)
    scale = torch.where(amax > 0, amax / SYMMETRIC_QMAX, torch.full_like(amax, UNIT_SCALE))
    inv_scale = 1.0 / scale
    return scale.contiguous(), inv_scale.contiguous()


def quantize_codes_reference(W: torch.Tensor, inv_scale: torch.Tensor) -> torch.Tensor:
    """Эталонная квантизация на torch (те же fp32-операции, что и в ядре).

    Вход: W (R, C) fp16; inv_scale float32 формы (1,) или (R,).
    Выход: (R, C) int32 коды в [-8, 7].
    Алгоритм: x = W * inv_scale -> clamp[-8, 7] -> floor(x + 0.5) (round half up).
    Ноль остаётся нулём: 0 * inv_scale = 0 -> floor(0.5) = 0.
    """
    x = W.to(torch.float32) * inv_scale.reshape(-1, 1)
    x = x.clamp(float(INT4_MIN), float(INT4_MAX))
    return torch.floor(x + ROUND_HALF).to(torch.int32)


def pack_int4_reference(codes: torch.Tensor, pack_bits: int) -> torch.Tensor:
    """Эталонная упаковка int4-кодов на torch.

    Вход: codes (R, C) целые значения в [-8, 7]; pack_bits ∈ {8, 32}.
    Выход: (R, ceil(C / ELEMS)) int8 / int32 (layout — см. docstring модуля).
    Трюк: nibble = code & 0xF (двоичное дополнение в 4 битах), затем
    word = sum(nibble << 4*e); результат считается в int64 и приводится к
    знаковому диапазону слова вычитанием 2**pack_bits.
    """
    epw = elems_per_word(pack_bits)
    rows, cols = codes.shape
    words_per_row = cdiv(cols, epw)
    pad = words_per_row * epw - cols
    codes = codes.to(torch.int64)
    if pad:
        codes = torch.nn.functional.pad(codes, (0, pad))
    nibbles = (codes & NIBBLE_MASK).reshape(rows, words_per_row, epw)
    shifts = torch.arange(epw, device=codes.device, dtype=torch.int64) * INT4_BITS
    word = (nibbles << shifts).sum(dim=-1)                  # беззнаковое слово [0, 2**pack_bits)
    modulus = 1 << pack_bits
    word = torch.where(word >= (modulus >> 1), word - modulus, word)
    return word.to(packed_torch_dtype(pack_bits))


def unpack_int4_reference(packed: torch.Tensor, cols: int, pack_bits: int) -> torch.Tensor:
    """Эталонная распаковка: упакованный тензор -> (R, cols) int32 знаковые коды.

    Вход: packed (R, W) int8/int32; cols — исходное число столбцов (без паддинга).
    Трюк: nibble = (word >> 4*e) & 0xF, знаковое расширение (nibble ^ 8) - 8.
    """
    epw = elems_per_word(pack_bits)
    rows, words_per_row = packed.shape
    word = packed.to(torch.int64) & ((1 << pack_bits) - 1)  # перевод в беззнаковое слово
    shifts = torch.arange(epw, device=packed.device, dtype=torch.int64) * INT4_BITS
    nibbles = (word.unsqueeze(-1) >> shifts) & NIBBLE_MASK
    codes = (nibbles ^ NIBBLE_SIGN_BIT) - NIBBLE_SIGN_BIT
    return codes.reshape(rows, words_per_row * epw)[:, :cols].to(torch.int32)


def dequantize_int4_to_fp16(W_packed: torch.Tensor, scale: torch.Tensor, shape, pack_bits: int = PACK_BITS_INT8) -> torch.Tensor:
    """Деквантизация int4 -> fp16 (чистый torch, для валидации).

    Вход:
      W_packed — (R, ceil(C / ELEMS)) int8 / int32;
      scale    — float32, форма (1,) (per-tensor) или (R,) (per-row);
      shape    — (R, C) исходная форма; pack_bits ∈ {8, 32}.
    Выход: (R, C) fp16, row-major. W ≈ code * scale.
    """
    rows, cols = shape
    if scale.numel() not in (1, rows):
        raise ValueError(f"scale.numel() должен быть 1 или {rows}, получено {scale.numel()}")
    codes = unpack_int4_reference(W_packed, cols, pack_bits)
    return (codes.to(torch.float32) * scale.to(torch.float32).reshape(-1, 1)).to(torch.float16)


def check_packed_memory(W: torch.Tensor, W_packed: torch.Tensor, pack_bits: int) -> None:
    """Проверяет выигрыш по памяти ровно в 4x.

    Если C кратно ELEMS (нет паддинга) — строгое равенство
        W_packed.numel() * W_packed.element_size() * 4 == W.numel() * W.element_size().
    Иначе проверяется точный размер с учётом выравнивания строк (паддинг < 1 слова на строку).
    """
    rows, cols = W.shape
    epw = elems_per_word(pack_bits)
    packed_bytes = W_packed.numel() * W_packed.element_size()
    if cols % epw == 0:
        if packed_bytes * EXPECTED_COMPRESSION != W.numel() * W.element_size():
            raise AssertionError(
                f"Память не уменьшилась ровно в {EXPECTED_COMPRESSION}x: "
                f"{packed_bytes} Б (int4) против {W.numel() * W.element_size()} Б (fp16)")
    else:
        expected = rows * cdiv(cols, epw) * (pack_bits // BITS_PER_BYTE)
        if packed_bytes != expected:
            raise AssertionError(f"Неверный размер упаковки: {packed_bytes} Б, ожидалось {expected} Б")

In [3]:
# Ядро квантизации fp16 -> int4 + обёртка
"""Triton-квантизация fp16 -> int4 с упаковкой в int8 (2 значения) или int32 (8 значений)."""

import torch
import triton
import triton.language as tl


# --------------------------------------------------------------------------
# Константы ядра (tl.constexpr — единственный способ читать глобалы внутри @triton.jit)
# --------------------------------------------------------------------------
_QMIN_F = tl.constexpr(float(INT4_MIN))
_QMAX_F = tl.constexpr(float(INT4_MAX))
_NIBBLE_MASK = tl.constexpr(NIBBLE_MASK)
_NIBBLE_BITS = tl.constexpr(INT4_BITS)
_ROUND_HALF = tl.constexpr(ROUND_HALF)

# --------------------------------------------------------------------------
# Параметры запуска по умолчанию (обоснование)
# --------------------------------------------------------------------------
# Одна программа обрабатывает ~2048 исходных fp16-элементов (4 КиБ чтения):
#  * при num_warps=4 (128 потоков) это 16 элементов/поток — хватает для скрытия
#    латентности HBM, но регистровое давление мало (тайл [BLOCK, ELEMS] в fp32);
#  * для pack=8 это BLOCK_SIZE=1024 слов, для pack=32 — 256 слов.
QUANT_ELEMS_PER_PROGRAM = 2048
QUANT_NUM_WARPS = 4


@triton.jit
def quantize_fp16_to_int4_kernel(
    w_ptr,                      # *fp16, (R, C) row-major
    inv_scale_ptr,              # *fp32, (1,) при PER_ROW=False или (R,) при PER_ROW=True
    out_ptr,                    # *int8 / *int32, (R, WORDS_PER_ROW) row-major
    n_cols,                     # C
    words_per_row,              # ceil(C / ELEMS)
    total_words,                # R * words_per_row
    PER_ROW: tl.constexpr,      # per-row (True) или per-tensor (False) scale
    ELEMS: tl.constexpr,        # значений int4 в слове: 2 (int8) или 8 (int32)
    PACK_BITS: tl.constexpr,    # 8 или 32 — ширина выходного слова
    BLOCK_SIZE: tl.constexpr,   # сколько выходных слов обрабатывает одна программа
):
    """Квантизует fp16 -> int4 (симметрично) и упаковывает nibble'ы в слова.

    Назначение: out[r, j] = pack( clamp(round(W[r, j*ELEMS + e] * inv_scale), -8, 7) for e in 0..ELEMS-1 ).
    Входы: W — fp16 (R, C) row-major; inv_scale — fp32 (1,) или (R,).
    Выход: int8 (PACK_BITS=8) или int32 (PACK_BITS=32), форма (R, ceil(C / ELEMS)).
    Layout: каждая строка независимо дополняется нулями до кратности ELEMS; элемент e
    слова лежит в битах [4e, 4e+4): чётный -> младшие 4 бита, нечётный -> старшие.
    Ограничения: ELEMS * 4 == PACK_BITS; BLOCK_SIZE и ELEMS — степени двойки.
    Трюки: nibble = q & 0xF (двоичное дополнение в 4 битах); упаковка = tl.sum от
    (nibble << 4e) — биты не пересекаются, поэтому сумма равна побитовому OR
    (переполнение int32 на старшем nibble'е — корректный wrap-around);
    floor(x + 0.5) реализован через усечение к нулю + tl.where (без libdevice).
    """
    pid = tl.program_id(axis=0)
    # int64, чтобы не переполнить индексы на больших матрицах
    word_idx = pid.to(tl.int64) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)   # [BLOCK]
    word_mask = word_idx < total_words                                      # хвост сетки

    row = word_idx // words_per_row                                         # строка слова
    word_in_row = word_idx - row * words_per_row                            # номер слова в строке
    elem = tl.arange(0, ELEMS)                                              # [ELEMS]

    # Исходный столбец каждого элемента: [BLOCK, ELEMS]
    col = word_in_row[:, None] * ELEMS + elem[None, :]
    # Маска: слово существует И столбец не выходит за C (паддинг строки -> нули)
    in_bounds = word_mask[:, None] & (col < n_cols)

    x = tl.load(w_ptr + row[:, None] * n_cols + col, mask=in_bounds, other=0.0).to(tl.float32)

    if PER_ROW:
        inv_scale = tl.load(inv_scale_ptr + row, mask=word_mask, other=0.0)   # [BLOCK]
        x = x * inv_scale[:, None]
    else:
        x = x * tl.load(inv_scale_ptr)                                        # скаляр

    # clamp в float до округления: результат гарантированно в [-8, 7]
    x = tl.minimum(tl.maximum(x, _QMIN_F), _QMAX_F)

    # floor(x + 0.5): усечение к нулю, затем поправка для отрицательных нецелых
    shifted = x + _ROUND_HALF
    truncated = shifted.to(tl.int32)
    q = tl.where(shifted < truncated.to(tl.float32), truncated - 1, truncated)  # int32 в [-8, 7]

    # Упаковка: двоичное дополнение в 4 битах, сдвиг на 4*e, суммирование по ELEMS
    nibble = q & _NIBBLE_MASK                                               # [BLOCK, ELEMS], 0..15
    bit_shift = elem * _NIBBLE_BITS                                         # [ELEMS]: 0, 4, 8, ...
    word = tl.sum(nibble << bit_shift[None, :], axis=1)                     # [BLOCK] int32

    if PACK_BITS == 8:
        # приведение int32 -> int8 отбрасывает старшие биты (нужные 8 бит уже в слове)
        tl.store(out_ptr + word_idx, word.to(tl.int8), mask=word_mask)
    else:
        tl.store(out_ptr + word_idx, word, mask=word_mask)


def quantize_fp16_to_int4(W: torch.Tensor, pack_bits: int = PACK_BITS_INT8, per_row: bool = False,
                          block_size: int = None, num_warps: int = QUANT_NUM_WARPS):
    """Квантизует fp16-матрицу в int4 и упаковывает (Triton).

    Вход:
      W         — (M, N) torch.float16, CUDA, row-major (не-contiguous будет скопирован);
      pack_bits — 8 (2 значения в int8) или 32 (8 значений в int32);
      per_row   — False: scale = max|W| / 7 на весь тензор (форма (1,));
                  True:  отдельный scale на каждую строку (форма (M,));
      block_size — число упакованных слов на программу (степень двойки);
                  по умолчанию QUANT_ELEMS_PER_PROGRAM // ELEMS;
      num_warps — число варпов (по умолчанию 4).
    Выход: (W_packed, scale, shape):
      W_packed — (M, ceil(N / ELEMS)) int8 / int32;
      scale    — float32, (1,) или (M,);
      shape    — (M, N) для деквантизации.
    Ограничения: строки дополняются нулями до кратности ELEMS (N % 2 == 0 для pack=8,
    N % 8 == 0 для pack=32 — условие «ровно 4x» без паддинга). Свойства: ноль -> 0,
    диапазон кодов [-8, 7], округление half-up.
    """
    if not (W.is_cuda and W.dtype == torch.float16 and W.dim() == 2):
        raise ValueError("W должен быть 2D torch.float16 тензором на CUDA")
    if W.numel() == 0:
        raise ValueError("пустая матрица не поддерживается")
    W = W.contiguous()
    rows, cols = W.shape
    epw = elems_per_word(pack_bits)
    words_per_row = cdiv(cols, epw)
    total_words = rows * words_per_row

    if block_size is None:
        block_size = max(QUANT_ELEMS_PER_PROGRAM // epw, 1)
    if block_size & (block_size - 1):
        raise ValueError("block_size должен быть степенью двойки")

    scale, inv_scale = compute_scale(W, per_row)
    out = torch.empty((rows, words_per_row), dtype=packed_torch_dtype(pack_bits), device=W.device)

    grid = (triton.cdiv(total_words, block_size),)
    quantize_fp16_to_int4_kernel[grid](
        W, inv_scale, out, cols, words_per_row, total_words,
        PER_ROW=per_row, ELEMS=epw, PACK_BITS=pack_bits, BLOCK_SIZE=block_size,
        num_warps=num_warps,
    )
    check_packed_memory(W, out, pack_bits)
    return out, scale, (rows, cols)

In [4]:
# Ядро матмула bf16 x int4 + обёртка
"""Triton-матмул bf16 x int4 -> bf16: out = X @ W4^T (с масштабом scale)."""

import torch
import triton
import triton.language as tl


# --------------------------------------------------------------------------
# Константы ядра
# --------------------------------------------------------------------------
INT32_BITS = 32
NIBBLE_TOP_SHIFT = INT32_BITS - INT4_BITS      # 28: nibble выравнивается к старшим битам int32
_NIBBLE_BITS = tl.constexpr(INT4_BITS)
_NIBBLE_TOP_SHIFT = tl.constexpr(NIBBLE_TOP_SHIFT)

# --------------------------------------------------------------------------
# Параметры запуска по умолчанию (обоснование)
# --------------------------------------------------------------------------
MATMUL_MIN_TILE = 16          # tl.dot требует размеров тайла >= 16 по каждой оси
MATMUL_MAX_BLOCK_M = 64       # 64x64 fp32-аккумулятор = 32 рег./поток при 4 варпах: без спиллов
MATMUL_BLOCK_N = 64           # ширина тайла по N для M > MATMUL_SKINNY_M
MATMUL_BLOCK_N_SKINNY = 32    # для «тонких» M (декодинг) — больше программ, лучше загрузка SM
MATMUL_SKINNY_M = 16
MATMUL_BLOCK_K = 64           # кратно ELEMS (2 и 8); 64 bf16 = 128 Б на строку X — полная кэш-линия
MATMUL_NUM_WARPS = 4          # тайл 64x64 покрывается 4 варпами без избыточности
MATMUL_NUM_STAGES = 3         # 3 стадии программного конвейера; в shared-памяти T4 (64 КиБ) помещается
BF16_MIN_CAPABILITY_MAJOR = 8  # tensor cores с bf16 — начиная с Ampere (sm_80)


@triton.jit
def matmul_bf16_x_int4_kernel(
    x_ptr,                      # *bf16, (M, K)
    w_ptr,                      # *int8 / *int32, (N, WORDS_PER_ROW) — упакованные веса
    scale_ptr,                  # *fp32, (1,) при PER_ROW=False или (N,) при PER_ROW=True
    out_ptr,                    # *bf16, (M, N)
    M, N, K,
    stride_xm, stride_xk,
    stride_wn, stride_ww,
    stride_om, stride_on,
    PER_ROW: tl.constexpr,      # scale по N (True) или один на тензор (False)
    ELEMS: tl.constexpr,        # значений int4 в слове: 2 (int8) или 8 (int32)
    UPCAST_FP32: tl.constexpr,  # True для GPU < sm_80: dot в fp32 вместо bf16
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,      # обязан быть кратен ELEMS
):
    """out[m, n] = scale[n] * sum_k X[m, k] * code(W4[n, k]), аккумулятор в fp32.

    Входы: X — bf16 (M, K) row-major; W4 — упакованные int4 (N, ceil(K / ELEMS)),
    int8 или int32, row-major, строки дополнены нулевыми nibble'ами; scale — fp32.
    Выход: bf16 (M, N) row-major.
    Ограничения: BLOCK_K % ELEMS == 0, BLOCK_M/N/K >= 16 и степени двойки; M, N, K — любые
    (границы закрыты масками). Для UPCAST_FP32 tl.dot идёт по FMA/TF32-пути.
    Трюки: знаковое расширение nibble'а без ветвлений —
        code = (word << (28 - 4e)) >> 28   (арифметический сдвиг int32),
    что эквивалентно lo = (packed << 4) >> 4 и hi = packed >> 4 из int8-случая, но
    работает и для int32. Каждый K-индекс адресует своё слово k // ELEMS (повторные
    чтения одного слова попадают в L1) — это даёт простой, портабельный по версиям
    Triton код без tl.join/interleave.
    """
    pid_n = tl.program_id(axis=0)
    pid_m = tl.program_id(axis=1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    m_mask = offs_m < M
    n_mask = offs_n < N

    # Положение nibble'а внутри слова; не зависит от итерации, т.к. BLOCK_K % ELEMS == 0
    word_in_block = offs_k // ELEMS                              # [BLOCK_K]: индекс слова в блоке
    bit_shift = (offs_k % ELEMS) * _NIBBLE_BITS                  # [BLOCK_K]: 0, 4, 8, ...
    left_shift = -bit_shift + _NIBBLE_TOP_SHIFT                  # 28 - 4e

    offs_m64 = offs_m.to(tl.int64)
    offs_n64 = offs_n.to(tl.int64)
    x_ptrs = x_ptr + offs_m64[:, None] * stride_xm + offs_k[None, :] * stride_xk          # [BM, BK]
    w_ptrs = w_ptr + offs_n64[None, :] * stride_wn + word_in_block[:, None] * stride_ww   # [BK, BN]

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k0 in range(0, K, BLOCK_K):
        k_mask = (k0 + offs_k) < K
        x = tl.load(x_ptrs, mask=m_mask[:, None] & k_mask[None, :], other=0.0)
        packed = tl.load(w_ptrs, mask=k_mask[:, None] & n_mask[None, :], other=0)

        # Распаковка: сдвиг влево выносит нужный nibble в биты 28..31, арифметический
        # сдвиг вправо возвращает его на место с расширением знака
        w = (packed.to(tl.int32) << left_shift[:, None]) >> _NIBBLE_TOP_SHIFT
        w = tl.where(k_mask[:, None], w, 0)                      # хвост K — строго нули

        if UPCAST_FP32:
            acc = tl.dot(x.to(tl.float32), w.to(tl.float32), acc)
        else:
            acc = tl.dot(x, w.to(tl.bfloat16), acc)              # int4 в bf16 представим точно

        x_ptrs += BLOCK_K * stride_xk
        w_ptrs += (BLOCK_K // ELEMS) * stride_ww

    # Эпилог: масштаб (broadcast по N) и запись в bf16
    if PER_ROW:
        scale = tl.load(scale_ptr + offs_n, mask=n_mask, other=0.0)
        acc = acc * scale[None, :]
    else:
        acc = acc * tl.load(scale_ptr)

    out_ptrs = out_ptr + offs_m64[:, None] * stride_om + offs_n64[None, :] * stride_on
    tl.store(out_ptrs, acc.to(tl.bfloat16), mask=m_mask[:, None] & n_mask[None, :])


def select_matmul_config(M: int) -> dict:
    """Эвристика выбора тайлов по M (значения обоснованы в константах модуля).

    Вход: M — число строк X. Выход: dict(block_m, block_n, block_k).
    M <= 16 (декодинг, memory-bound): BLOCK_M=16, BLOCK_N=32 — больше программ по N.
    M > 16: BLOCK_M = ближайшая степень двойки (<= 64), BLOCK_N=64.
    """
    block_m = min(max(triton.next_power_of_2(M), MATMUL_MIN_TILE), MATMUL_MAX_BLOCK_M)
    block_n = MATMUL_BLOCK_N_SKINNY if M <= MATMUL_SKINNY_M else MATMUL_BLOCK_N
    return dict(block_m=block_m, block_n=block_n, block_k=MATMUL_BLOCK_K)


def matmul_bf16_x_int4(x: torch.Tensor, w_packed: torch.Tensor, scale: torch.Tensor, shape,
                       pack_bits: int = PACK_BITS_INT8, block_m: int = None, block_n: int = None,
                       block_k: int = None, num_warps: int = MATMUL_NUM_WARPS,
                       num_stages: int = MATMUL_NUM_STAGES) -> torch.Tensor:
    """Считает X @ W4^T, где W4 — упакованная int4-матрица (N, K) (Triton).

    Вход:
      x        — (M, K) torch.bfloat16, CUDA (не-contiguous будет скопирован);
      w_packed — (N, ceil(K / ELEMS)) int8 / int32 из quantize_fp16_to_int4;
      scale    — float32, (1,) (per-tensor) или (N,) (per-row по N);
      shape    — (N, K) исходная форма W;
      pack_bits — 8 или 32 (должен совпадать с упаковкой);
      block_m/n/k, num_warps, num_stages — параметры запуска (None -> эвристика).
    Выход: (M, N) torch.bfloat16.
    Ограничения: block_k кратен ELEMS и >= 16, степени двойки; на GPU < sm_80 dot
    выполняется в fp32 (bf16 tensor cores отсутствуют) — корректно, но медленнее.
    """
    n_out, k_dim = shape
    epw = elems_per_word(pack_bits)
    if not (x.is_cuda and x.dtype == torch.bfloat16 and x.dim() == 2):
        raise ValueError("x должен быть 2D torch.bfloat16 тензором на CUDA")
    if x.shape[1] != k_dim:
        raise ValueError(f"K не совпадает: x.shape[1]={x.shape[1]}, shape[1]={k_dim}")
    if w_packed.dtype != packed_torch_dtype(pack_bits) or tuple(w_packed.shape) != (n_out, cdiv(k_dim, epw)):
        raise ValueError(f"w_packed: ожидалось {packed_torch_dtype(pack_bits)} {(n_out, cdiv(k_dim, epw))}, "
                         f"получено {w_packed.dtype} {tuple(w_packed.shape)}")
    if scale.dtype != torch.float32 or scale.numel() not in (1, n_out):
        raise ValueError("scale: float32 с numel 1 или N")
    if x.shape[0] == 0 or n_out == 0 or k_dim == 0:
        raise ValueError("пустые размеры не поддерживаются")

    x = x.contiguous()
    scale = scale.to(x.device).contiguous()
    m_dim = x.shape[0]

    cfg = select_matmul_config(m_dim)
    block_m = block_m or cfg["block_m"]
    block_n = block_n or cfg["block_n"]
    block_k = block_k or cfg["block_k"]
    if block_k % epw or min(block_m, block_n, block_k) < MATMUL_MIN_TILE:
        raise ValueError(f"block_k должен быть кратен {epw}, все блоки >= {MATMUL_MIN_TILE}")
    for blk in (block_m, block_n, block_k):
        if blk & (blk - 1):
            raise ValueError("block_m/n/k должны быть степенями двойки")

    per_row = scale.numel() != 1
    upcast = torch.cuda.get_device_capability(x.device)[0] < BF16_MIN_CAPABILITY_MAJOR
    out = torch.empty((m_dim, n_out), dtype=torch.bfloat16, device=x.device)

    grid = (triton.cdiv(n_out, block_n), triton.cdiv(m_dim, block_m))
    matmul_bf16_x_int4_kernel[grid](
        x, w_packed, scale, out,
        m_dim, n_out, k_dim,
        x.stride(0), x.stride(1),
        w_packed.stride(0), w_packed.stride(1),
        out.stride(0), out.stride(1),
        PER_ROW=per_row, ELEMS=epw, UPCAST_FP32=upcast,
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k,
        num_warps=num_warps, num_stages=num_stages,
    )
    return out

In [5]:
# Тесты. Запуск: ячейка выполняет run_all_tests() и печатает отчёт по группам.
import itertools
import math

import torch

# --------------------------------------------------------------------------
# Константы тестов
# --------------------------------------------------------------------------
TEST_SEED = 0
DEVICE = "cuda"
EDGE_SIZES = (1, 17, 128, 1000)                     # «некруглые» размеры для M, N, K
QUANT_SHAPES = ((1, 1), (17, 33), (128, 128), (1000, 17), (64, 1000))
ACCURACY_SHAPE = (1024, 1024)
WEIGHT_STD = 0.1                                     # W = randn * 0.1
MATMUL_ATOL = 1e-2
MATMUL_RTOL = 1e-2
PREDICTED_ERR_TOL = 0.10                             # допуск к теоретической оценке ошибки квантизации
UNIFORM_NOISE_DIVISOR = 12.0                         # дисперсия равномерного шума округления = шаг^2 / 12
PER_ROW_MATMUL_SHAPES = ((1, 17, 1000), (17, 1000, 128), (128, 128, 17), (1000, 1, 1))  # (M, N, K)


def make_weights(rows: int, cols: int) -> torch.Tensor:
    """Гауссовы fp16-веса N(0, 0.1^2) формы (rows, cols) на CUDA (генерация в fp32, затем fp16)."""
    return (torch.randn(rows, cols, device=DEVICE, dtype=torch.float32) * WEIGHT_STD).to(torch.float16)


# ==========================================================================
# Тесты. Каждый тест — функция с assert; возвращает короткую строку с деталями.
# ==========================================================================
def test_pack_roundtrip(rows: int, cols: int, pack_bits: int) -> str:
    """Что проверяем: pack -> unpack возвращает исходные коды (все значения от -8 до 7).
    Зачем: ловит ошибки знака и паддинга строк в эталонных torch-функциях (CPU)."""
    codes = torch.arange(INT4_MIN, INT4_MAX + 1, dtype=torch.int32).repeat(rows * cols)[: rows * cols].reshape(rows, cols)
    packed = pack_int4_reference(codes, pack_bits)
    assert torch.equal(unpack_int4_reference(packed, cols, pack_bits), codes), "коды изменились после pack -> unpack"
    return f"{rows}x{cols}"


def test_pack_layout_known_vector() -> str:
    """Что проверяем: на вектор [1, -2, 3, -4, 5, -6, 7, -8] упаковка даёт ровно те слова,
    которые мы считаем вручную: чётный индекс -> младшие 4 бита, нечётный -> старшие, знак в
    двоичном дополнении. Зачем: фиксирует формат хранения, от которого зависит матмул."""
    codes_list = [1, -2, 3, -4, 5, -6, 7, -8]
    codes = torch.tensor([codes_list], dtype=torch.int32)
    nibbles = [c & NIBBLE_MASK for c in codes_list]

    expected_int8 = []
    for i in range(0, len(nibbles), 2):
        byte = nibbles[i] | (nibbles[i + 1] << INT4_BITS)
        expected_int8.append(byte - 256 if byte >= 128 else byte)
    got_int8 = pack_int4_reference(codes, PACK_BITS_INT8).tolist()
    assert got_int8 == [expected_int8], f"int8: {got_int8} != {[expected_int8]}"

    word = sum(n << (INT4_BITS * i) for i, n in enumerate(nibbles))
    word = word - (1 << 32) if word >= (1 << 31) else word
    got_int32 = pack_int4_reference(codes, PACK_BITS_INT32).tolist()
    assert got_int32 == [[word]], f"int32: {got_int32} != {[[word]]}"
    return f"int8 -> {expected_int8}, int32 -> {word}"


def test_quant_kernel_equals_torch(shape, pack_bits: int, per_row: bool) -> str:
    """Что проверяем: упакованные данные Triton-ядра совпадают бит-в-бит с независимой
    torch-реализацией (scale -> округление -> упаковка). Зачем: основной тест корректности
    ядра квантизации, включая паддинг строк и хвосты блоков."""
    rows, cols = shape
    W = make_weights(rows, cols)
    packed, scale, out_shape = quantize_fp16_to_int4(W, pack_bits=pack_bits, per_row=per_row)
    assert out_shape == (rows, cols), f"shape: {out_shape}"
    _, inv_scale = compute_scale(W, per_row)
    codes_ref = quantize_codes_reference(W, inv_scale)
    expected = pack_int4_reference(codes_ref, pack_bits)
    assert packed.dtype == expected.dtype and packed.shape == expected.shape, "не совпали dtype/shape упаковки"
    assert torch.equal(packed, expected), "упакованные данные отличаются от эталона"
    assert INT4_MIN <= int(codes_ref.min()) and int(codes_ref.max()) <= INT4_MAX, "код вне диапазона int4"
    return f"{rows}x{cols}"


def test_quant_error_is_within_one_step(pack_bits: int, per_row: bool) -> str:
    """Что проверяем на W ~ N(0, 0.1^2), 1024x1024 (quantize -> dequantize):
    1) |W - W_deq| <= scale.max() (allclose с atol=scale.max(), rtol=0);
    2) нули остаются нулями (симметричная квантизация);
    3) относительная ошибка по Фробениусу совпадает с теорией: шум округления равномерный,
       поэтому ||err|| / ||W|| ≈ sqrt(mean(scale^2) / 12) / rms(W) (допуск 10 %)."""
    rows, cols = ACCURACY_SHAPE
    W = make_weights(rows, cols)
    W[::7, ::5] = 0                                          # заведомые нули
    packed, scale, shape = quantize_fp16_to_int4(W, pack_bits=pack_bits, per_row=per_row)
    W_deq = dequantize_int4_to_fp16(packed, scale, shape, pack_bits)

    assert torch.allclose(W, W_deq, atol=scale.max().item(), rtol=0), "ошибка больше одного шага scale"
    assert bool((W_deq[W == 0] == 0).all()), "ноль перестал быть нулём"

    W32, D32 = W.float(), W_deq.float()
    measured = ((W32 - D32).norm() / W32.norm()).item()
    noise_ms = (scale.float() ** 2).mean().item() / UNIFORM_NOISE_DIVISOR
    predicted = math.sqrt(noise_ms / (W32 ** 2).mean().item())
    assert abs(measured - predicted) / predicted < PREDICTED_ERR_TOL, \
        f"ошибка {measured:.4f} не совпала с теорией {predicted:.4f}"
    return f"rel_fro={measured:.4f} (теория {predicted:.4f}), max|err|={(W32 - D32).abs().max().item():.4f} <= scale={scale.max().item():.4f}"


def test_memory_is_4x_smaller(pack_bits: int) -> str:
    """Что проверяем: packed.numel() * element_size * 4 == W.numel() * element_size (1024x1024),
    а для матрицы 17x33 (строки с паддингом) — точный ожидаемый размер.
    Зачем: гарантия выигрыша по памяти."""
    W = make_weights(*ACCURACY_SHAPE)
    packed, _, _ = quantize_fp16_to_int4(W, pack_bits=pack_bits)
    packed_bytes, fp16_bytes = packed.numel() * packed.element_size(), W.numel() * W.element_size()
    assert packed_bytes * 4 == fp16_bytes, f"{packed_bytes} Б * 4 != {fp16_bytes} Б"
    W_odd = make_weights(17, 33)
    packed_odd, _, _ = quantize_fp16_to_int4(W_odd, pack_bits=pack_bits)
    check_packed_memory(W_odd, packed_odd, pack_bits)
    return f"{fp16_bytes} Б (fp16) -> {packed_bytes} Б (int4)"


def test_matmul_matches_reference(M: int, N: int, K: int, pack_bits: int, per_row: bool = False) -> float:
    """Что проверяем: Triton-матмул X @ W4^T совпадает с X.float() @ dequant(W).T
    (atol = rtol = 1e-2). Возвращает max abs error. Зачем: основной тест матмула,
    включая M, N, K не кратные размерам блоков."""
    X = torch.randn(M, K, device=DEVICE, dtype=torch.float32).to(torch.bfloat16)
    W = make_weights(N, K)
    packed, scale, shape = quantize_fp16_to_int4(W, pack_bits=pack_bits, per_row=per_row)
    out = matmul_bf16_x_int4(X, packed, scale, shape, pack_bits=pack_bits)
    assert out.dtype == torch.bfloat16 and out.shape == (M, N), f"выход {out.dtype} {tuple(out.shape)}"
    ref = X.float() @ dequantize_int4_to_fp16(packed, scale, shape, pack_bits).float().T
    max_err = (out.float() - ref).abs().max().item()
    assert torch.allclose(out.float(), ref, atol=MATMUL_ATOL, rtol=MATMUL_RTOL), \
        f"M={M} N={N} K={K} pack={pack_bits} per_row={per_row}: max_abs_err={max_err:.5f}"
    return max_err


def test_rejects_bad_inputs() -> str:
    """Что проверяем: неверные dtype и pack_bits отвергаются ValueError, а не падением ядра."""
    W = make_weights(8, 16)
    packed, scale, shape = quantize_fp16_to_int4(W)
    X_fp32 = torch.randn(4, 16, device=DEVICE, dtype=torch.float32)
    bad_calls = {
        "x не bf16": lambda: matmul_bf16_x_int4(X_fp32, packed, scale, shape),
        "W не fp16": lambda: quantize_fp16_to_int4(W.float()),
        "pack_bits=16": lambda: quantize_fp16_to_int4(W, pack_bits=16),
    }
    for label, call in bad_calls.items():
        try:
            call()
        except ValueError:
            continue
        raise AssertionError(f"«{label}» не вызвал ValueError")
    return "3 некорректных вызова отвергнуты"


# ==========================================================================
# Запуск и отчёт
# ==========================================================================
def run_group(title: str, cases) -> bool:
    """Запускает набор тестов и печатает одну строку-итог (+ строки по упавшим).

    cases — список (функция, аргументы). Если тест вернул число, оно считается ошибкой
    (берётся максимум); если строку — печатается как деталь (для одиночных тестов)."""
    failures, details, worst = [], [], None
    for fn, args in cases:
        try:
            result = fn(*args)
            if isinstance(result, float):
                worst = result if worst is None else max(worst, result)
            elif isinstance(result, str):
                details.append(result)
        except Exception as exc:                              # собираем все падения, не останавливаясь
            failures.append(f"{fn.__name__}{args}: {type(exc).__name__}: {exc}")
    passed = len(cases) - len(failures)
    mark = "OK  " if not failures else "FAIL"
    extra = f"  worst max_abs_err = {worst:.5f}" if worst is not None else ""
    print(f"[{mark}] {title}: {passed}/{len(cases)}{extra}")
    if len(cases) <= 4:                                       # для маленьких групп показываем детали
        for d in details:
            print(f"        {d}")
    elif details and not failures:
        print(f"        например: {details[0]}")
    for f in failures:
        print(f"        ✗ {f}")
    return not failures


def run_all_tests() -> None:
    """Запускает все группы тестов; при любом падении бросает AssertionError."""
    torch.manual_seed(TEST_SEED)
    torch.cuda.manual_seed_all(TEST_SEED)
    packs = SUPPORTED_PACK_BITS
    ok = [
        run_group("Эталон: pack -> unpack возвращает исходные коды",
                  [(test_pack_roundtrip, (*s, pb)) for pb in packs for s in ((1, 1), (3, 5), (4, 8), (7, 17))]),
        run_group("Эталон: формат упаковки на известном векторе", [(test_pack_layout_known_vector, ())]),
        run_group("Квантизация: Triton == torch-эталон (бит-в-бит), 5 форм x 2 pack x 2 режима scale",
                  [(test_quant_kernel_equals_torch, (s, pb, pr)) for s in QUANT_SHAPES for pb in packs for pr in (False, True)]),
    ]
    for pb, pr in itertools.product(packs, (False, True)):
        ok.append(run_group(f"Квантизация: ошибка и нули (1024x1024), pack={pb}, per_row={pr}",
                            [(test_quant_error_is_within_one_step, (pb, pr))]))
    for pb in packs:
        ok.append(run_group(f"Память: ровно 4x, pack={pb}", [(test_memory_is_4x_smaller, (pb,))]))
    for pb in packs:
        ok.append(run_group(f"Матмул: M, N, K ∈ {set(EDGE_SIZES)}, pack={pb} (64 комбинации)",
                            [(test_matmul_matches_reference, (m, n, k, pb)) for m, n, k in itertools.product(EDGE_SIZES, repeat=3)]))
    for pb in packs:
        ok.append(run_group(f"Матмул: per-row scale, pack={pb}",
                            [(test_matmul_matches_reference, (*mnk, pb, True)) for mnk in PER_ROW_MATMUL_SHAPES]))
    ok.append(run_group("Валидация входов", [(test_rejects_bad_inputs, ())]))

    if not all(ok):
        raise AssertionError("Есть упавшие тесты — см. строки FAIL выше")
    print("\nВСЕ ТЕСТЫ ПРОЙДЕНЫ")


if __name__ == "__main__":
    run_all_tests()

[OK  ] Эталон: pack -> unpack возвращает исходные коды: 8/8
        например: 1x1
[OK  ] Эталон: формат упаковки на известном векторе: 1/1
        int8 -> [-31, -61, -91, -121], int32 -> -2019179551
[OK  ] Квантизация: Triton == torch-эталон (бит-в-бит), 5 форм x 2 pack x 2 режима scale: 20/20
        например: 1x1
[OK  ] Квантизация: ошибка и нули (1024x1024), pack=8, per_row=False: 1/1
        rel_fro=0.2191 (теория 0.2224), max|err|=0.0380 <= scale=0.0760
[OK  ] Квантизация: ошибка и нули (1024x1024), pack=8, per_row=True: 1/1
        rel_fro=0.1422 (теория 0.1442), max|err|=0.0364 <= scale=0.0729
[OK  ] Квантизация: ошибка и нули (1024x1024), pack=32, per_row=False: 1/1
        rel_fro=0.2124 (теория 0.2154), max|err|=0.0368 <= scale=0.0736
[OK  ] Квантизация: ошибка и нули (1024x1024), pack=32, per_row=True: 1/1
        rel_fro=0.1416 (теория 0.1437), max|err|=0.0361 <= scale=0.0724
[OK  ] Память: ровно 4x, pack=8: 1/1
        2097152 Б (fp16) -> 524288 Б (int4)
[OK  ] Память: ров

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[OK  ] Матмул: M, N, K ∈ {128, 1, 17, 1000}, pack=8 (64 комбинации): 64/64  worst max_abs_err = 0.03410
[OK  ] Матмул: M, N, K ∈ {128, 1, 17, 1000}, pack=32 (64 комбинации): 64/64  worst max_abs_err = 0.03619
[OK  ] Матмул: per-row scale, pack=8: 4/4  worst max_abs_err = 0.01543
[OK  ] Матмул: per-row scale, pack=32: 4/4  worst max_abs_err = 0.01603
[OK  ] Валидация входов: 1/1
        3 некорректных вызова отвергнуты

ВСЕ ТЕСТЫ ПРОЙДЕНЫ


In [6]:
# Бенчмарк: время (cuda.Event) и пропускная способность, сравнение с наивным torch
import statistics

import torch


SEED = 0
BENCH_WARMUP = 10                    # прогрев: компиляция и кэши
BENCH_ITERS = 50                     # число замеров; берём медиану
BYTES_PER_GB = 1e9
MS_PER_S = 1e3
QUANT_SHAPE = (4096, 4096)           # (N, K) типичного слоя LLM
MATMUL_M_LIST = (1, 16, 2048)        # декодинг (memory-bound) и prefill (compute-bound)
BF16_BYTES = 2
FP32_BYTES = 4
SWEEP_M = 16
SWEEP_CONFIGS = [                    # (block_n, block_k, num_warps, num_stages)
    (32, 64, 4, 3), (32, 128, 4, 3), (64, 64, 4, 3), (64, 128, 4, 3),
    (64, 128, 8, 3), (128, 64, 8, 3), (32, 64, 2, 4), (64, 64, 4, 2),
]


def time_ms(fn, warmup: int = BENCH_WARMUP, iters: int = BENCH_ITERS) -> float:
    """Медианное время вызова fn() в мс по torch.cuda.Event.

    Вход: fn — функция без аргументов (запускает GPU-работу); warmup/iters — число прогревов/замеров.
    Выход: медиана времени одного вызова, мс.
    """
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(iters):
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end))
    return statistics.median(samples)


def gbps(num_bytes: int, ms: float) -> float:
    """Пропускная способность в ГБ/с: байты / время."""
    return num_bytes / BYTES_PER_GB / (ms / MS_PER_S)


def bench_quantize() -> None:
    """Сравнивает Triton-квантизацию с torch-пайплайном (коды + упаковка) для pack=8 и pack=32."""
    rows, cols = QUANT_SHAPE
    W = (torch.randn(rows, cols, device="cuda") * 0.1).to(torch.float16)
    print(f"== Квантизация fp16 -> int4, W {QUANT_SHAPE} ==")
    print(f"{'pack':>5} {'impl':>8} {'ms':>9} {'GB/s':>9} {'speedup':>8}")
    for pb in SUPPORTED_PACK_BITS:
        packed, _, _ = quantize_fp16_to_int4(W, pack_bits=pb)
        moved = W.numel() * W.element_size() + packed.numel() * packed.element_size()   # чтение W + запись packed
        t_tri = time_ms(lambda: quantize_fp16_to_int4(W, pack_bits=pb))

        def naive():
            _, inv = compute_scale(W, False)
            return pack_int4_reference(quantize_codes_reference(W, inv), pb)
        t_ref = time_ms(naive, warmup=3, iters=10)
        print(f"{pb:>5} {'triton':>8} {t_tri:9.4f} {gbps(moved, t_tri):9.1f} {t_ref / t_tri:7.1f}x")
        print(f"{pb:>5} {'torch':>8} {t_ref:9.4f} {gbps(moved, t_ref):9.1f} {'1.0x':>8}")
    print("(время Triton включает вычисление scale: abs().amax() в torch)")


def bench_matmul() -> None:
    """Сравнивает Triton-матмул с наивным X @ dequant(W).T и с cuBLAS на bf16-весах."""
    n_dim, k_dim = QUANT_SHAPE
    W = (torch.randn(n_dim, k_dim, device="cuda") * 0.1).to(torch.float16)
    W_bf16 = W.to(torch.bfloat16)
    print(f"\n== Матмул bf16 x int4 -> bf16, W ({n_dim}, {k_dim}) ==")
    print(f"{'pack':>5} {'M':>5} {'impl':>16} {'ms':>9} {'GB/s':>9} {'vs naive':>9}")
    for pb in SUPPORTED_PACK_BITS:
        packed, scale, shape = quantize_fp16_to_int4(W, pack_bits=pb)
        for m_dim in MATMUL_M_LIST:
            X = torch.randn(m_dim, k_dim, device="cuda").to(torch.bfloat16)
            moved = (X.numel() * BF16_BYTES + packed.numel() * packed.element_size() + m_dim * n_dim * BF16_BYTES)
            t_tri = time_ms(lambda: matmul_bf16_x_int4(X, packed, scale, shape, pack_bits=pb))
            t_naive = time_ms(lambda: X @ dequantize_int4_to_fp16(packed, scale, shape, pb).to(torch.bfloat16).T,
                              warmup=3, iters=10)
            moved_bf16 = X.numel() * BF16_BYTES + W_bf16.numel() * BF16_BYTES + m_dim * n_dim * BF16_BYTES
            t_cublas = time_ms(lambda: X @ W_bf16.T)
            print(f"{pb:>5} {m_dim:>5} {'triton int4':>16} {t_tri:9.4f} {gbps(moved, t_tri):9.1f} {t_naive / t_tri:8.1f}x")
            print(f"{pb:>5} {m_dim:>5} {'torch naive':>16} {t_naive:9.4f} {gbps(moved, t_naive):9.1f} {'1.0x':>9}")
            print(f"{pb:>5} {m_dim:>5} {'cuBLAS bf16 W':>16} {t_cublas:9.4f} {gbps(moved_bf16, t_cublas):9.1f} {'-':>9}")
    print("GB/s считается по «полезным» байтам: X + упакованные W + выход (для cuBLAS — W в bf16).")


def sweep_matmul_configs() -> None:
    """Небольшой перебор (BLOCK_N, BLOCK_K, num_warps, num_stages) при M=SWEEP_M, pack=8.

    Нужен, чтобы обосновать дефолты на конкретном GPU; печатает варианты по возрастанию времени.
    """
    n_dim, k_dim = QUANT_SHAPE
    W = (torch.randn(n_dim, k_dim, device="cuda") * 0.1).to(torch.float16)
    packed, scale, shape = quantize_fp16_to_int4(W, pack_bits=PACK_BITS_INT8)
    X = torch.randn(SWEEP_M, k_dim, device="cuda").to(torch.bfloat16)
    results = []
    for block_n, block_k, warps, stages in SWEEP_CONFIGS:
        try:
            ms = time_ms(lambda: matmul_bf16_x_int4(X, packed, scale, shape, block_n=block_n, block_k=block_k,
                                                    num_warps=warps, num_stages=stages), warmup=5, iters=20)
            results.append((ms, block_n, block_k, warps, stages))
        except Exception as exc:                              # конфиг может не влезть в shared memory
            print(f"  BLOCK_N={block_n} BLOCK_K={block_k} warps={warps} stages={stages}: пропущен ({type(exc).__name__})")
    print(f"\n== Перебор конфигов, M={SWEEP_M}, pack=8 ==")
    for ms, block_n, block_k, warps, stages in sorted(results):
        print(f"  BLOCK_N={block_n:>3} BLOCK_K={block_k:>3} warps={warps} stages={stages}: {ms:.4f} мс")


def run_benchmarks() -> None:
    """Запускает все бенчмарки."""
    torch.manual_seed(SEED)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    bench_quantize()
    bench_matmul()
    sweep_matmul_configs()


run_benchmarks()

GPU: Tesla T4
== Квантизация fp16 -> int4, W (4096, 4096) ==
 pack     impl        ms      GB/s  speedup
    8   triton    1.0784      38.9     9.8x
    8    torch   10.5383       4.0     1.0x
   32   triton    0.6628      63.3    12.8x
   32    torch    8.4807       4.9     1.0x
(время Triton включает вычисление scale: abs().amax() в torch)

== Матмул bf16 x int4 -> bf16, W (4096, 4096) ==
 pack     M             impl        ms      GB/s  vs naive
    8     1      triton int4    1.5065       5.6      5.6x
    8     1      torch naive    8.3892       1.0      1.0x
    8     1    cuBLAS bf16 W    0.1965     170.9         -
    8    16      triton int4    1.4128       6.1      6.3x
    8    16      torch naive    8.8916       1.0      1.0x
    8    16    cuBLAS bf16 W    0.7644      44.2         -
    8  2048      triton int4   42.2323       1.0      0.8x
    8  2048      torch naive   32.6013       1.3      1.0x
    8  2048    cuBLAS bf16 W   26.5759       2.5         -
   32     1     

## Итоги

**Что проверяют тесты (ячейка 6)** — каждая группа печатает одну строку `OK` / `FAIL`:
- Эталонная упаковка: round-trip всех кодов −8…7 и точный формат на известном векторе (чётный индекс → младшие 4 бита, знак в двоичном дополнении), оба `pack_bits`.
- Ядро квантизации побитово совпадает с независимой torch-реализацией для форм `(1,1)`, `(17,33)`, `(128,128)`, `(1000,17)`, `(64,1000)`, обоих `pack_bits` и обоих режимов scale.
- Точность на `W = randn · 0.1`, 1024×1024: `allclose(W, W_deq, atol=scale.max(), rtol=0)`, нули остаются нулями, ошибка по Фробениусу согласуется с теорией равномерного шума округления.
- Память: `packed.numel() · element_size · 4 == W.numel() · element_size` для int8 и int32; для строк с паддингом проверяется точный размер.
- Матмул против `X.float() @ dequant(W).T` с `atol = rtol = 1e-2`: все 64 комбинации `M, N, K ∈ {1, 17, 128, 1000}` × `pack_bits ∈ {8, 32}`, плюс per-row scale и отказ на некорректных входах.

**Что показывает бенчмарк (ячейка 7)**
- Время и GB/s обоих ядер против наивного torch-варианта, а также против cuBLAS на bf16-весах.
- Перебор конфигураций `(BLOCK_N, BLOCK_K, num_warps, num_stages)` при `M = 16` — основание для дефолтов матмула.

**Ограничения**
- Каждый K-индекс читает своё упакованное слово (`k // ELEMS`), повторные чтения идут из L1. Так код проще и переносим между версиями Triton, но пропускная способность по весам ниже теоретической. Следующий шаг — загрузка слов один раз и распаковка через `tl.join`/`tl.interleave`.
- Scale вычисляется в torch (`abs().amax()`), а не в отдельном ядре.
- Нет autotune: размеры тайлов выбираются эвристикой `select_matmul_config`.
- На sm < 80 используется fp32-путь `tl.dot`.